In [ ]:
import sys
sys.path.append('../utils_function/')
import utils_function as uf
import common_functions as cf
import pandas as pd
import numpy as np
import os
from collections import defaultdict

In [ ]:
def get_optimal_params(performance_dict):
    """
    从性能字典中找出平均相关性最高的参数
    
    参数:
    performance_dict (dict): 包含不同参数对应相关性列表的字典
    
    返回:
    float: 最优参数值
    """
    best_lambda = None
    best_avg_corr = -float('inf')  # 初始化为负无穷

    # 遍历每个 lambda 及其对应的相关性列表
    for lambda_val, corr_list in performance_dict.items():
        # 计算当前 lambda 的平均相关性
        avg_corr = sum(corr_list) / len(corr_list)
        
        # 更新最优参数
        if avg_corr > best_avg_corr:
            best_avg_corr = avg_corr
            best_lambda = lambda_val
    
    return best_lambda

In [ ]:
animal_list = ['mouse', 'marmoset', 'macaque', 'human', 'marmoset']
dataset_key_list = ['gamanut', 'majka', 'markov40', 'lariviere', 'V4']
performance_dict = {}
optimal_params_dict = {}
for animal, dataset_key in zip(animal_list, dataset_key_list):
    print(f'Running for {animal} and {dataset_key}')

    performance_dict_this_animal = defaultdict(list)

    model_container = cf.MetaModelContainer(uf.AxonGrowthModel, cf.pj('../../results/para_search', f'{animal}_{dataset_key}'))

    for model in model_container:
        performance_dict_this_animal[model.get_unique_Lambda()].append(model.get_data_model_correlation())
    
    performance_dict[f'{animal}_{dataset_key}'] = performance_dict_this_animal
    optimal_params_dict[f'{animal}_{dataset_key}'] = {'Lambda': get_optimal_params(performance_dict_this_animal)}
    print(performance_dict_this_animal)
    print(f'Optimal lambda for {animal}_{dataset_key}: {optimal_params_dict[f"{animal}_{dataset_key}"]}')
    print(f'Optimal average correlation: {max([sum(v)/len(v) for v in performance_dict_this_animal.values()])}')